# Retrospective Variant-A inventory sealing

This is an **unexecuted, source-read-only operator notebook** pinned to merged main commit `09605a0ce5a3c14d3e19ea7c719405d5cc5d35b3`. It computes the exact complete `R/train` and `R/validation` inventory SHA-256 values required by `stage2_variant_a_qualification_colab.ipynb`, plus the Gate 0.1 result-file SHA-256.

It uses the merged notebook's reviewed `D:\MRI_Field_2026\Data` to Colab-root remapping and byte-identical inventory arithmetic. Every `P` record is classified and excluded before any source-file hashing; array loading is prohibited. The only output is one atomic, no-clobber candidate-inventory JSON in a dedicated inventory-sealing output root that must be disjoint from the future scientific qualification output root. It performs no fitting, VAE construction or inference, qualification, latent-bank construction, training, paired evaluation, or prospective execution.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/GuillermoTafoya/MRIxFields.git"
PINNED_COMMIT = "09605a0ce5a3c14d3e19ea7c719405d5cc5d35b3"
EXPECTED_PARENTS = (
    "d3f7c3b0123b76c187478c7775c40d33e574daf7",
    "2c3a6a6eb329bf43803626d19d2c96106a9ca13e",
)
REPO_DIR = Path("/content/MRIxFields-inventory-seal-09605a0")

if REPO_DIR.exists():
    raise FileExistsError(
        f"Refusing to reuse or alter {REPO_DIR}. Start a fresh Colab runtime or remove it manually."
    )
subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", PINNED_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_DIR, check=True)

def git_text(*args: str) -> str:
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

actual_head = git_text("rev-parse", "HEAD")
origin_main = git_text("rev-parse", "refs/remotes/origin/main")
parent_line = git_text("rev-list", "--parents", "-n", "1", "HEAD").split()
actual_parents = tuple(parent_line[1:])
status = git_text("status", "--porcelain")
detached = subprocess.run(
    ["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO_DIR, capture_output=True
).returncode != 0
if actual_head != PINNED_COMMIT:
    raise RuntimeError(f"Detached HEAD mismatch: {actual_head}, expected {PINNED_COMMIT}.")
if actual_parents != EXPECTED_PARENTS:
    raise RuntimeError(f"Merge parents changed: {actual_parents} != {EXPECTED_PARENTS}.")
if status or not detached:
    raise RuntimeError(f"Checkout must be detached and clean; status={status!r}, detached={detached}.")
pinned_is_origin_main_ancestor = subprocess.run(
    ["git", "merge-base", "--is-ancestor", PINNED_COMMIT, "refs/remotes/origin/main"],
    cwd=REPO_DIR,
).returncode == 0
if not pinned_is_origin_main_ancestor:
    raise RuntimeError("Pinned reviewed commit is not an ancestor of the observed origin/main.")
print({"head": actual_head, "observed_origin_main": origin_main, "parents": actual_parents, "detached": detached, "clean": True})

In [ ]:
import importlib
import json
import tomllib

# Use the same declared dependency set as the merged qualification operator.
with (REPO_DIR / "pyproject.toml").open("rb") as handle:
    project = tomllib.load(handle)["project"]
optional = project["optional-dependencies"]
requirements = list(dict.fromkeys(
    [*project["dependencies"], *optional["evaluation"], *optional["official-evaluation"]]
))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements],
    check=True,
)

SOURCE_DIR = str(REPO_DIR / "src")
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = SOURCE_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in tuple(sys.modules):
    if module_name == "fieldbridge" or module_name.startswith("fieldbridge."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge

package_file = Path(fieldbridge.__file__).resolve()
if REPO_DIR not in package_file.parents:
    raise RuntimeError(f"FieldBridge import escaped the pinned checkout: {package_file}.")
if git_text("status", "--porcelain"):
    raise RuntimeError("Dependency setup changed the pinned checkout.")
print(json.dumps({"fieldbridge_import": str(package_file), "requirements": requirements}, indent=2))

## External read-only inputs and isolated candidate output

Use the immutable original split-v3, the retrospective data root containing the remapped files, and the reviewed Gate 0.1 result JSON. Choose a new inventory-sealing output root that is outside the repository, retrospective data root, and future Variant-A qualification output root. The notebook writes only `stage2_variant_a_candidate_inventory_v1.json` there.

In [ ]:
import re
from google.colab import drive

drive.mount("/content/drive")

def required_external_path(prompt: str) -> Path:
    value = input(prompt).strip()
    if not value:
        raise ValueError(f"A path is required for: {prompt}")
    return Path(value).expanduser()

FROZEN_SPLIT_V3_JSON = required_external_path("Immutable frozen split-v3 JSON: " )
RETROSPECTIVE_DATA_ROOT = required_external_path("External retrospective data root: " )
GATE01_RESULT_JSON = required_external_path("Reviewed Gate 0.1 result JSON: " )
INVENTORY_SEALING_OUTPUT_ROOT = required_external_path("New inventory-sealing output root: " )
QUALIFICATION_OUTPUT_ROOT = required_external_path("Future scientific qualification output root (separation check only): " )
REVIEWED_WINDOWS_SOURCE_ROOT = r"D:\MRI_Field_2026\Data"
EXPECTED_SPLIT_V3_SHA256 = input("Expected immutable split-v3 file SHA-256: " ).strip().lower()
if re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SPLIT_V3_SHA256) is None:
    raise ValueError("split-v3 requires an exact lowercase SHA-256.")

In [ ]:
# Seal only source-file identities and complete retrospective inventories; never load arrays.
from collections import Counter
from pathlib import PureWindowsPath
from unittest import mock
import copy
import hashlib

import fieldbridge.cli as fb_cli
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.photometry_factorization import (
    all_photometry_domain_labels,
    assert_variant_a_external_path,
    sha256_file,
    sha256_json,
    sha256_text,
    write_json_atomic,
)
from fieldbridge.data.vae_splits import (
    VaeSplits,
    load_vae_splits,
    vae_splits_fingerprint,
    vae_splits_recovery_fingerprint_v3,
)

if git_text("rev-parse", "HEAD") != PINNED_COMMIT or git_text("status", "--porcelain"):
    raise RuntimeError("Pinned checkout is no longer clean. Stop before reading external inputs.")
FROZEN_SPLIT_V3_JSON = assert_variant_a_external_path(FROZEN_SPLIT_V3_JSON, repo_root=REPO_DIR)
RETROSPECTIVE_DATA_ROOT = assert_variant_a_external_path(RETROSPECTIVE_DATA_ROOT, repo_root=REPO_DIR)
GATE01_RESULT_JSON = assert_variant_a_external_path(GATE01_RESULT_JSON, repo_root=REPO_DIR)
INVENTORY_SEALING_OUTPUT_ROOT = assert_variant_a_external_path(INVENTORY_SEALING_OUTPUT_ROOT, repo_root=REPO_DIR)
QUALIFICATION_OUTPUT_ROOT = assert_variant_a_external_path(QUALIFICATION_OUTPUT_ROOT, repo_root=REPO_DIR)
if not FROZEN_SPLIT_V3_JSON.is_file():
    raise FileNotFoundError(f"Missing immutable split-v3: {FROZEN_SPLIT_V3_JSON}")
if not RETROSPECTIVE_DATA_ROOT.is_dir():
    raise NotADirectoryError(f"Missing retrospective data root: {RETROSPECTIVE_DATA_ROOT}")
if not GATE01_RESULT_JSON.is_file():
    raise FileNotFoundError(f"Missing Gate 0.1 result: {GATE01_RESULT_JSON}")

def require_disjoint_trees(left: Path, right: Path, label: str) -> None:
    left_resolved = left.resolve()
    right_resolved = right.resolve()
    for child, parent in ((left_resolved, right_resolved), (right_resolved, left_resolved)):
        try:
            child.relative_to(parent)
        except ValueError:
            continue
        raise ValueError(f"{label} must be disjoint: {left_resolved} and {right_resolved}")

require_disjoint_trees(INVENTORY_SEALING_OUTPUT_ROOT, QUALIFICATION_OUTPUT_ROOT, "Inventory and qualification output roots")
require_disjoint_trees(INVENTORY_SEALING_OUTPUT_ROOT, RETROSPECTIVE_DATA_ROOT, "Inventory output and retrospective data roots")
if INVENTORY_SEALING_OUTPUT_ROOT.exists() and not INVENTORY_SEALING_OUTPUT_ROOT.is_dir():
    raise NotADirectoryError(f"Inventory output root is not a directory: {INVENTORY_SEALING_OUTPUT_ROOT}")
if INVENTORY_SEALING_OUTPUT_ROOT.exists() and any(INVENTORY_SEALING_OUTPUT_ROOT.iterdir()):
    raise FileExistsError("Inventory-sealing output root must be new or empty.")
INVENTORY_SEALING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_root = RETROSPECTIVE_DATA_ROOT
original_split_bytes = FROZEN_SPLIT_V3_JSON.read_bytes()
original_split_file_sha256 = hashlib.sha256(original_split_bytes).hexdigest()
if original_split_file_sha256 != EXPECTED_SPLIT_V3_SHA256:
    raise RuntimeError("Immutable split-v3 SHA-256 mismatch.")
original_splits = load_vae_splits(FROZEN_SPLIT_V3_JSON)
original_split_membership = vae_splits_fingerprint(original_splits)
original_split_recovery = vae_splits_recovery_fingerprint_v3(original_splits)
original_split_payload = json.loads(original_split_bytes.decode("utf-8"))
if not isinstance(original_split_payload, dict):
    raise ValueError("Original split-v3 root must be a JSON object.")

reviewed_old_root = PureWindowsPath(REVIEWED_WINDOWS_SOURCE_ROOT)
if not reviewed_old_root.is_absolute():
    raise ValueError("Reviewed Windows source root must be absolute.")
resolved_data_root = RETROSPECTIVE_DATA_ROOT.resolve(strict=True)

def remap_reviewed_windows_path(raw_path: str) -> Path:
    source = PureWindowsPath(str(raw_path))
    source_parts = tuple(part.casefold() for part in source.parts)
    root_parts = tuple(part.casefold() for part in reviewed_old_root.parts)
    if not source.is_absolute() or source_parts[:len(root_parts)] != root_parts:
        raise ValueError(
            f"Frozen split path is outside reviewed Windows root {reviewed_old_root}: {raw_path}"
        )
    relative_parts = source.parts[len(reviewed_old_root.parts):]
    if not relative_parts or any(part in {"", ".", ".."} for part in relative_parts):
        raise ValueError(f"Frozen split path has an unsafe relative suffix: {raw_path}")
    mapped = resolved_data_root.joinpath(*relative_parts).resolve(strict=False)
    try:
        mapped.relative_to(resolved_data_root)
    except ValueError as exc:
        raise ValueError(f"Remapped source escapes retrospective root: {mapped}") from exc
    return mapped

operational_split_payload = copy.deepcopy(original_split_payload)
mapping_entries = []
for split_name in ("train", "validation", "test"):
    records_payload = operational_split_payload.get("splits", {}).get(split_name)
    if not isinstance(records_payload, list):
        raise ValueError(f"Original split is missing record list {split_name!r}.")
    for record_payload in records_payload:
        if not isinstance(record_payload, dict) or "image_path" not in record_payload:
            raise ValueError(f"Malformed {split_name} record in original split.")
        old_path = str(record_payload["image_path"])
        new_path = remap_reviewed_windows_path(old_path)
        record_payload["image_path"] = str(new_path)
        mapping_entries.append({
            "split": split_name,
            "record_identity": str(record_payload.get("case_id", "")),
            "old_path": old_path,
            "new_path": str(new_path),
        })
mapping_entries.sort(key=lambda item: (item["split"], item["record_identity"], item["old_path"]))
mapping_identity_sha256 = sha256_json(mapping_entries)
operational_metadata = dict(operational_split_payload.get("metadata", {}))
operational_metadata["colab_path_remap"] = {
    "contract": "stage2-colab-windows-root-remap-v1",
    "original_split_file_sha256": original_split_file_sha256,
    "original_membership_fingerprint": original_split_membership,
    "original_recovery_fingerprint_v3": original_split_recovery,
    "reviewed_old_root": str(reviewed_old_root),
    "operational_new_root": str(resolved_data_root),
    "remapped_record_count": len(mapping_entries),
    "mapping_identity_sha256": mapping_identity_sha256,
}
operational_split_payload["metadata"] = operational_metadata

def records_from_operational_payload(split_name: str):
    return tuple(record_from_mapping(item) for item in operational_split_payload["splits"][split_name])

operational_candidate = VaeSplits(
    train=records_from_operational_payload("train"),
    validation=records_from_operational_payload("validation"),
    test=records_from_operational_payload("test"),
    seed=int(operational_split_payload["seed"]),
    fractions=tuple(float(value) for value in operational_split_payload["fractions"]),
    metadata=operational_metadata,
)
operational_split_membership = vae_splits_fingerprint(operational_candidate)
operational_split_recovery = vae_splits_recovery_fingerprint_v3(operational_candidate)
operational_split_payload["fingerprint"] = operational_split_membership
operational_split_payload["recovery_fingerprint_v3"] = operational_split_recovery
if operational_split_membership != original_split_membership:
    raise RuntimeError("Path remapping changed split membership.")
original_assignments = {
    name: tuple(sorted(record.case_id for record in original_splits.records_for(name)))
    for name in ("train", "validation", "test")
}
operational_assignments = {
    name: tuple(sorted(record.case_id for record in operational_candidate.records_for(name)))
    for name in ("train", "validation", "test")
}
if operational_assignments != original_assignments:
    raise RuntimeError("Path remapping changed train/validation/test assignments.")
splits = operational_candidate

array_load_attempts = 0
def forbidden_array_load(*args, **kwargs):
    global array_load_attempts
    array_load_attempts += 1
    raise AssertionError("Array loading is forbidden during inventory sealing.")
with mock.patch.object(fb_cli, "load_volume", side_effect=forbidden_array_load):
    fit_records, fit_excluded = fb_cli._select_variant_a_retrospective_records(
        splits.train, split="train"
    )
    qualification_records, qualification_excluded = fb_cli._select_variant_a_retrospective_records(
        splits.validation, split="validation"
    )
    classified = {
        split_name: tuple(fb_cli._classify_variant_a_split_record(record) for record in records)
        for split_name, records in {
            "train": splits.train, "validation": splits.validation, "test": splits.test
        }.items()
    }
if array_load_attempts != 0:
    raise AssertionError("Production cohort selection attempted to load an array.")

def identity_set(items):
    return {item.case_identity for item in items}
for split_name, selected, excluded in (
    ("train", fit_records, fit_excluded),
    ("validation", qualification_records, qualification_excluded),
):
    expected_r = {item.case_identity for item in classified[split_name] if item.cohort == "R"}
    expected_p = {item.case_identity for item in classified[split_name] if item.cohort == "P"}
    selected_ids = {str(record.case_id) for record in selected}
    excluded_ids = {str(item["record_identity"]) for item in excluded}
    if selected_ids != expected_r or excluded_ids != expected_p or selected_ids & excluded_ids:
        raise AssertionError(f"Fail-closed R/P selection proof failed for {split_name}.")
if not fit_records or not qualification_records:
    raise ValueError("Both R/train fitting and R/validation qualification require records.")
expected_domains = set(all_photometry_domain_labels())
fit_domain_counts = dict(sorted(Counter(record.domain.label for record in fit_records).items()))
qualification_domain_counts = dict(sorted(Counter(record.domain.label for record in qualification_records).items()))
if set(fit_domain_counts) != expected_domains or set(qualification_domain_counts) != expected_domains:
    raise ValueError("Inventory sealing requires all 15 domains in both eligible roles.")

inventory_records = []
for split_name, records in (("train", fit_records), ("validation", qualification_records)):
    for record in records:
        cohort_identity = fb_cli._classify_variant_a_split_record(record)
        if cohort_identity.cohort != "R":
            raise AssertionError("Only production-classified R records may enter an inventory.")
        source = Path(record.image_path)
        if not source.is_absolute():
            raise ValueError(
                f"Frozen split source paths must be absolute because the official CLI uses them verbatim: {source}"
            )
        source = source.resolve(strict=True)
        try:
            relative = source.relative_to(data_root.resolve(strict=True))
        except ValueError as exc:
            raise ValueError(f"Eligible source escapes retrospective root: {source}") from exc
        inventory_records.append({
            "split": split_name,
            "record_identity": str(record.case_id),
            "record_identity_sha256": sha256_text(str(record.case_id)),
            "subject_group_identity": cohort_identity.subject_group_identity,
            "domain": record.domain.label,
            "relative_source_path": relative.as_posix(),
            "source_path_identity_sha256": sha256_text(str(record.image_path)),
            "source_bytes": source.stat().st_size,
            "source_file_sha256": sha256_file(source),
        })
inventory_records.sort(key=lambda item: (item["split"], item["domain"], item["record_identity"]))
fit_inventory_records = [item for item in inventory_records if item["split"] == "train"]
qualification_inventory_records = [
    item for item in inventory_records if item["split"] == "validation"
]
fit_inventory_sha256 = sha256_json(fit_inventory_records)
qualification_inventory_sha256 = sha256_json(qualification_inventory_records)
gate01_result_file_sha256 = sha256_file(GATE01_RESULT_JSON)

def compact_source_identities(records):
    return [
        {
            "record_identity": item["record_identity"],
            "source_path_identity_sha256": item["source_path_identity_sha256"],
            "source_file_sha256": item["source_file_sha256"],
        }
        for item in records
    ]

fit_source_identities = compact_source_identities(fit_inventory_records)
qualification_source_identities = compact_source_identities(qualification_inventory_records)
fit_subject_group_counts = dict(sorted(Counter(
    item["subject_group_identity"] for item in fit_inventory_records
).items()))
qualification_subject_group_counts = dict(sorted(Counter(
    item["subject_group_identity"] for item in qualification_inventory_records
).items()))
candidate = {
    "contract": "stage2-variant-a-candidate-inventory-v1",
    "status": "candidate-requires-external-review",
    "code_commit": PINNED_COMMIT,
    "operator_inputs": {
        "EXPECTED_SPLIT_V3_SHA256": original_split_file_sha256,
        "EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256": fit_inventory_sha256,
        "FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256": qualification_inventory_sha256,
        "EXPECTED_GATE01_RESULT_SHA256": gate01_result_file_sha256,
    },
    "computed_candidate_hashes": {
        "complete_R_train_inventory_sha256": fit_inventory_sha256,
        "complete_R_validation_inventory_sha256": qualification_inventory_sha256,
        "gate01_result_file_sha256": gate01_result_file_sha256,
    },
    "split_provenance": {
        "original_file_sha256": original_split_file_sha256,
        "original_membership_fingerprint": original_split_membership,
        "original_recovery_fingerprint_v3": original_split_recovery,
        "operational_membership_fingerprint": operational_split_membership,
        "operational_recovery_fingerprint_v3": operational_split_recovery,
        "reviewed_old_root": str(reviewed_old_root),
        "operational_new_root": str(resolved_data_root),
        "remapped_record_count": len(mapping_entries),
        "mapping_identity_sha256": mapping_identity_sha256,
        "membership_preserved": operational_split_membership == original_split_membership,
        "all_assignments_preserved": operational_assignments == original_assignments,
        "original_file_remained_byte_identical": FROZEN_SPLIT_V3_JSON.read_bytes() == original_split_bytes,
    },
    "R_train": {
        "complete_record_count": len(fit_inventory_records),
        "domain_counts": fit_domain_counts,
        "all_15_domains_present": set(fit_domain_counts) == expected_domains,
        "subject_group_counts": fit_subject_group_counts,
        "excluded_P_identities_and_reasons": list(fit_excluded),
        "source_identities": fit_source_identities,
        "source_identities_sha256": sha256_json(fit_source_identities),
    },
    "R_validation": {
        "complete_record_count": len(qualification_inventory_records),
        "domain_counts": qualification_domain_counts,
        "all_15_domains_present": set(qualification_domain_counts) == expected_domains,
        "subject_group_counts": qualification_subject_group_counts,
        "excluded_P_identities_and_reasons": list(qualification_excluded),
        "source_identities": qualification_source_identities,
        "source_identities_sha256": sha256_json(qualification_source_identities),
    },
    "safety_proof": {
        "classification_completed_before_source_hashing": True,
        "array_load_attempts": array_load_attempts,
        "prospective_source_files_opened": 0,
        "endpoint_or_performance_selection_used": False,
        "fitting_performed": False,
        "vae_inference_performed": False,
        "qualification_performed": False,
        "latent_bank_constructed": False,
        "training_performed": False,
        "inventory_output_disjoint_from_qualification_output": True,
    },
}
if FROZEN_SPLIT_V3_JSON.read_bytes() != original_split_bytes:
    raise RuntimeError("Immutable split-v3 changed during inventory sealing.")
candidate["candidate_payload_sha256"] = sha256_json(candidate)
CANDIDATE_JSON = INVENTORY_SEALING_OUTPUT_ROOT / "stage2_variant_a_candidate_inventory_v1.json"
write_json_atomic(CANDIDATE_JSON, candidate)
if sha256_file(CANDIDATE_JSON) != hashlib.sha256(
    (json.dumps(candidate, indent=2, sort_keys=True, allow_nan=False) + "\n").encode("utf-8")
).hexdigest():
    raise RuntimeError("Candidate inventory publication bytes changed.")
print(json.dumps(candidate, indent=2, sort_keys=True))
print(f"SAVED: {CANDIDATE_JSON}")
print("STOP: externally review the candidate JSON. Do not run scientific qualification here.")

## Intentional final stop

The preceding cell is the end of this notebook. Copy the four values under `operator_inputs` into the merged qualification operator only after external review. Do not add fitting, VAE, qualification, latent-bank, training, or prospective-data cells here.